# DepthWizard DAV2/V1 — Kaggle 2x T4

**Accelerator: `GPU T4 x2`. Internet: on. `HF_TOKEN` added under Add-ons -> Secrets.**

Same runbook as V4_Kaggle, one variable changed: the encoder is **Depth Anything
V2 Base** (DINOv2 ViT-B/14 + its own pretrained DPT neck) instead of DINOv3
ViT-L/16 SAT-493M + a randomly-initialised trunk.

Two consequences on this card:

* The DAv2 checkpoint is **ungated** — `HF_TOKEN` is still needed, but only for
  GAMUS.
* 86 M encoder parameters instead of 303 M, so `--grad_checkpoint_encoder false`
  and `--batch_size 6` instead of the forced-on checkpointing at batch 2 that
  ViT-L needed.  Global batch stays 24 (2 ranks x 6 x grad_accum 2), so the LR
  schedule is still comparable to v3/v4 and the backbone is the only variable.

**Licence:** DAv2-Base is CC-BY-NC-4.0 (non-commercial).  Only DAv2-Small is
Apache-2.0.  Note this in the SIH writeup; the fallback is
`--encoder_model_id depth-anything/Depth-Anything-V2-Small-hf`.

Start the real run with **Save Version -> Save & Run All (Commit)**.  The
headless commit is what gets the full ~12 h and persists `/kaggle/working` as
the notebook's output; an interactive session idles out long before the run ends.


In [ ]:
# 1. Code + deps
!git clone --depth 1 https://github.com/AbhayKale332/DepthWizard.git /kaggle/working/DepthWizard 2>/dev/null || true
%cd /kaggle/working/DepthWizard/FineTunning/DAV2_V1
!pip install -q -r requirements.txt
!bash run_kaggle.sh check


In [ ]:
# 2. HF token.  GAMUS is gated; the DAv2 encoder is NOT.
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))


In [ ]:
# 3. Point a single data_root at whatever was attached.
#    Attach (Add Input) the abhaydkale232 datasets: depthwizard-gamus,
#    depthwizard-synrs3d-g05, depthwizard-synrs3d-g1 (+ depthwizard-india-labeled).
#    They mount as /kaggle/input/datasets/abhaydkale232/<slug>/...; `link` finds
#    every index.json and exposes it as /kaggle/temp/dwdata/<source>/<split>.
#    Symlinks only -- nothing is copied, so this costs no quota and no time.
!bash run_kaggle.sh link

In [ ]:
# 4. Prove the checkpoint's geometry before spending 8 hours on it.
#    518 -> 37x37 tokens -> 296 px fusion output -> 259 px (H/2), against the
#    real published weights rather than against arithmetic in a docstring.
!DW_TEST_REAL_CKPT=1 python -m pytest -q tests/test_real_dav2.py


In [ ]:
# 5. Train.  Two processes, one per T4.
#    Session 1:  as-is.
#    Session 2:  append --resume /kaggle/input/<session-1-output>/outputs/dav2-v1/last_full.pt
#    Both sessions keep the SAME --max_minutes; only --session_minutes caps this one.
!bash run_kaggle.sh train


## First 200 steps: four things to check, from the filesystem

A green cell and a frozen output pane prove nothing (that lesson cost a Modal
run).  Open a terminal / use `!` and read the files.

```bash
tail -40 /kaggle/working/outputs/dav2-v1/run.log
```

1. **`train_loss` is finite.**  fp16 + GradScaler on a freshly-attached neck is
   the likeliest place for a NaN, and a trainer can report PASS with
   `train_loss = NaN`.
2. **`vram=NG`** (printed every 25 steps) leaves >= 3 GB headroom for the eval
   spike.  If it does not, back off in order: `--consistency_every 4`, then
   `--ema_decay 0`, then `--w_consistency 0`.  If there is spare headroom, go to
   `--batch_size 8 --grad_accum 2` -- raise the batch, do not cut grad_accum.
3. **img/s is meaningfully above V4_Kaggle's 2.1 img/s unfrozen.**  If it is
   not, the 86 M encoder is not buying what this profile assumes and the batch
   size should be re-derived from the measurement, not from the comment in
   `run_kaggle.sh`.
4. **The `[optim]` line's LR span covers all 12 blocks at usable values** --
   `llrd 0.85 ** 12` is ~14 % of `encoder_lr`, so expect a range of roughly
   8e-06..3e-04, not something ending in e-07.

## Scoring

Compare `test_gamus_test_*`, `dfc23_g050` and `india_labeled` against v4's
numbers in `FineTunning/V4_modal/Output/`.  **Never quote `final_*`** -- it is
scored on the very tiles `best.pt` was selected on.

Report per-stratum `flat_bias` / `tall_bias` explicitly.  That pair is the
diagnostic that identified v4's problem (flat_bias frozen at +1.08 m against
v3's +0.396 m), and it is the first place a backbone change should show up.

## If the session dies

`last_full.pt` in the committed output carries the optimiser, scaler, EMA,
teacher, RNG and elapsed minutes.  Attach that version as an input to a fresh
notebook and add `--resume /kaggle/input/<name>/outputs/dav2-v1/last_full.pt` to
cell 5.  `run.log` should show the LR continuing the cosine rather than
restarting it.
